# Demo - PySpark + Apache Sedona

Quick demo to verify the environment works and explore the normalized NY crime data.

In [ ]:
!pip install pyspark==3.4.2 apache-sedona==1.6.1

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SedonaDemo")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.sedona:sedona-spark-shaded-3.4_2.12:1.6.1"
    )
    .config(
        "spark.sql.extensions",
        "org.apache.sedona.sql.SedonaSqlExtensions"
    )
    .config(
        "spark.serializer",
        "org.apache.spark.serializer.KryoSerializer"
    )
    .config(
        "spark.kryo.registrator",
        "org.apache.sedona.core.serde.SedonaKryoRegistrator"
    )
    .getOrCreate()
)

## Verify Sedona

In [ ]:
# Test that Sedona spatial functions are available
spark.sql("SELECT ST_Point(1.0, 2.0) AS test_point").show()

## Load Normalized NY Crime Data

In [ ]:
df = spark.read.parquet("data/new_york/crimes_normalized/parquet")
print(f"Total records: {df.count():,}")
df.printSchema()

In [ ]:
df.show(5, truncate=30)

## Geospatial Queries with Sedona

In [ ]:
from pyspark.sql.functions import expr

df_geo = df.withColumn("geometry", expr("ST_Point(longitude, latitude)"))
df_geo.createOrReplaceTempView("crimes")

df_geo.select("complaint_id", "borough", "geometry").show(5, truncate=False)

In [ ]:
spark.sql("""
    SELECT 
        count(*) AS total,
        min(year) AS min_year,
        max(year) AS max_year
    FROM crimes
""").show()

In [ ]:
spark.sql("""
    SELECT borough, offense_level, count(*) AS total
    FROM crimes
    GROUP BY borough, offense_level
    ORDER BY total DESC
""").show()

In [ ]:
# Example: Find crimes within 1km of Times Square (40.7580, -73.9855)
spark.sql("""
    SELECT complaint_id, borough, offense_description, premise_type,
           ST_Distance(
               ST_Transform(geometry, 'epsg:4326', 'epsg:3857'),
               ST_Transform(ST_Point(-73.9855, 40.7580), 'epsg:4326', 'epsg:3857')
           ) AS distance_meters
    FROM crimes
    WHERE ST_Distance(
        ST_Transform(geometry, 'epsg:4326', 'epsg:3857'),
        ST_Transform(ST_Point(-73.9855, 40.7580), 'epsg:4326', 'epsg:3857')
    ) < 1000
    ORDER BY distance_meters
    LIMIT 10
""").show(truncate=30)

In [ ]:
spark.stop()